# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and analyze the *FAIR² Mental Health Survey* dataset using the `mlcroissant` library. 

### Dataset Source
The dataset is described and distributed using the Croissant format and accessed via the schema URL below.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --upgrade mlcroissant

## 1. Data Loading
Load the dataset's metadata and the record set structure using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset (the metadata and access to record sets)
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description from metadata
print(f"Dataset Title: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}")

## 2. Data Overview
List the available record sets, their `@id`s, and the available fields (columns) for each. All references are to be made explicitly with `@id` values.

In [ ]:
# List all record sets and their attributes (using @id, as per best practice)
print("Available record sets in this dataset (referenced by @id):\n")

record_sets = list(dataset.metadata.record_sets)
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet Name: {rs.name} | @id: {rs.id}")
print()

# For each record set, print its fields (columns) and their @ids
for rs in record_sets:
    print(f"RecordSet: {rs.name} (@id: {rs.id}) fields:")
    for field in rs.fields:
        column_ids = [c.id for c in (field.columns or [])]
        print(f"- Field: {field.name} | @id: {field.id} | column_ids: {column_ids}")
    print()

## 3. Data Extraction
Load records for one or more record sets of interest using their `@id`. Data for each record set is loaded into a pandas DataFrame for further processing and analysis.

> **Note:** All references to record sets and fields use their `@id`.

In [ ]:
# Choose one or more record sets to extract (use @id)
# First, show all available record set ids. For this dataset, we'll assume the main survey data is in the first record set.
main_record_set = record_sets[0]
main_record_set_id = main_record_set.id
print(f"Selected RecordSet @id: {main_record_set_id}")

dataframes = {}
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
dataframes[main_record_set_id] = df

print(f"Loaded DataFrame columns for RecordSet {main_record_set_id}:")
print(df.columns.tolist())

df.head()

## 4. Exploratory Data Analysis (EDA)
Perform basic data processing:
- Filtering records (e.g., by age or score)
- Normalizing a numeric column
- Grouping by a categorical field

> You may reference fields by their `@id` as printed in the overview above. We'll assume typical field ids such as `'age'`, `'PHQ-9_score'`, and `'gender'` (please adjust if schema uses different @ids).

In [ ]:
from mlcroissant._dataset.metadata import Field, RecordSet

# Utility: get @id for fields of interest
numeric_field_id = None
group_field_id = None
for field in main_record_set.fields:
    if 'phq' in field.name.lower() and 'score' in field.name.lower():
        numeric_field_id = field.id  # e.g., PHQ-9 total score
    if 'gender' in field.name.lower():
        group_field_id = field.id

if numeric_field_id is None:
    # fallback: use the first numeric column
    for col in df.columns:
        if df[col].dtype in [int, float]:
            numeric_field_id = col
            break
if group_field_id is None:
    for col in df.columns:
        if 'gender' in col.lower():
            group_field_id = col
            break

print(f"Using numeric field for EDA: {numeric_field_id}")

# 1. Filter records with PHQ-9 score greater than a threshold (e.g., moderately severe)
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered {filtered_df.shape[0]} records where {numeric_field_id} > {threshold}.")
else:
    filtered_df = df.copy()
    print('Numeric field not found, using all records.')

# 2. Normalize the numeric field for filtered records
if numeric_field_id in filtered_df.columns:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print('Numeric field not found for normalization.')

# 3. Group by gender (or another group field) and compute means
if group_field_id in filtered_df.columns and numeric_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df[[numeric_field_id]])
else:
    print('Could not group: one of the grouping fields was not found.')

## 5. Visualization
Visualize the chosen numeric field's distribution and relationship with the group field if available.

*Histogram of PHQ-9 scores and boxplot by gender as examples*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
if numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

if group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
- This notebook demonstrated loading and inspecting a Croissant-based dataset via `mlcroissant`.
- Record sets, their structure, and main fields were explored using their `@id`.
- Core EDA steps and visualizations were applied, showing potential for further statistical or machine learning analysis.

> Remember to verify field and record set `@id` values in the overview section if you apply more sophisticated processing.

*Thank you for using FAIR² data and mlcroissant!*